# LEC production GLM

Cross-validated, binned, full regressor set — the fit that supersedes every cached pickle.

**What changed, and why it matters** (all measured; see `code/ANATOMY_SPLIT.md` and
`code/w1_encoding_diagnostics.py`):

| change | effect |
|---|---|
| Binned aggregation instead of strided subsampling | held-out R² **+0.0009 → +0.0411** |
| Head-direction fix | all 36 HD columns were previously noise |
| Leave-one-session-out CV | in-sample CPD was inflated; noise data gave +0.002 in-sample, −0.003 held-out |
| Mixed reference coding | full rank **with** pokes (160/160), so betas stay interpretable |
| `attach_pokes` | without it the poke CPDs are exactly 0 and the design loses 2 ranks |
| Δr² alongside CPD | CPD overstates weak regressors up to **12×** when a dominant one remains |

Run order: **gates → configuration comparison → production fit → regional join**.

## 0. Setup and gates

Nothing downstream is trustworthy if these fail.

In [1]:
import os, sys, pickle, importlib
import numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('.'))

import glm_analysis_v2 as glm
import glm_cv as cv
import anatomy_split as asp
import w1_refit as w
import run_glm_batch as rb

pd.set_option('display.width', 200)
glm.apply_gridmaze_style()

In [2]:
# W0 gates. Re-run rather than trusting the cached pickle -- these are cheap and they are
# the only thing standing between a mapping bug and a figure.
import w0_gates
gates = w0_gates.main(strict=True)     # hard-asserts the 25/25 length gate

W0 GATES — anatomy split
unit_regions: 25 recdays

loading data_dic (3.8 GB, this takes a few minutes) ...
load_data_dic: 25 recdays from data_dic_lec.pkl
validate_data_dic: 25 recdays checked, 0 skipped (no sorting on disk), 0 mismatched
validate_tasks_against_pycontrol: 191 sessions checked, 0 unresolved, 0 mismatched

[gate 1] length gate: 25/25 recdays PASS

[gate 2] HD columns: 187 sessions with (T,2) HD_raw
  sessions where >50% of samples sit within 25.0 deg of +/-90: 187/187
  median |earL2earR - back2mid| across sessions: 84.8 deg
  pooled median frac_within_tol_of_90: 0.983
  25 session(s) without usable HD: {'no HD': 25}

[gate 2b] HD length: fixed HD matches Locs_raw in 187/187 sessions
  flattened length == 2T in 187/187
  under the OLD flatten(), median fraction of the HD TIMELINE covered: 0.500

[gate 3] depth ordering (median y_um per region; 0 = shank tip)

  ah08: shallow->deep by median y_um = ENTl-sup < ENTl-deep   [PASS, judged on 2 group(s) present in all recdays]

## 1. Configuration comparison

`{uniform, decile} × {250 ms, 500 ms}` on one recday per mouse, cross-validated, no
permutations. Pick one configuration, then fit it properly.

Bin width is **not** a resolution worry: median leg is 9.38 s, so one `goal_progress` decile
is 938 ms and a sample spans 0.27 progress bins at 250 ms, 0.53 at 500 ms. The cost of 500 ms
is at the short end — **15.7% of legs have <10 samples** (vs 1.4% at 250 ms), so they cannot
populate all ten deciles.

In [3]:
data_dic = glm.load_data_dic(validate=True, apply_exclusions=True, verbose=True)
n_pokes = glm.attach_pokes(data_dic, verbose=False)
print(f'attach_pokes: {n_pokes} session tables')

# one recday per mouse -- the first smoke test used three ah08 recdays, the lowest-firing
# mouse, and could not speak for the cohort
seen, recdays_cmp = set(), []
for r in sorted(data_dic):
    m = r.split('_')[0]
    if m not in seen:
        seen.add(m); recdays_cmp.append(r)
print(recdays_cmp)

load_data_dic: 25 recdays from data_dic_lec.pkl
validate_data_dic: 25 recdays checked, 0 skipped (no sorting on disk), 0 mismatched
validate_tasks_against_pycontrol: 191 sessions checked, 0 unresolved, 0 mismatched
attach_pokes: 191 session tables
['ah08_20250613_20250615', 'ah10_20250613_20250615', 'ly05_20250613_20250615', 'ly06_20250613_20250615', 'ly07_20250613_20250615']


In [4]:
REG = w.SECTIONS['all_regressors']['regressors']
results = {}

for width in (250, 500):
    for scheme in ('decile', 'uniform'):
        factor = width // 25
        out = glm.run_glm_analysis(
            recdays_cmp, data_dic, num_permutations=1,
            regressors_to_include=REG, compute_cpd=True,
            parameterization=w.choose_parameterization(REG),
            downsample_factor=factor, downsample_mode='bin',
            continuous_binning=scheme,
            cross_validate=True, cv_n_perm=0, cv_only=True,
            cv_center_within_sessions=True,
        )
        results[(width, scheme)] = out[-1]
        r2 = np.nanmean([np.nanmedian(v['r2_cv']) for v in out[-1].values()])
        print(f'  {width} ms / {scheme:8s}  median r2_cv {r2:+.5f}')

Processing recording days:   0%|          | 0/5 [00:00<?, ?it/s]


ah08_20250613_20250615
  Design matrix: 27321 rows × 160 cols (regressors: ['place', 'task_state', 'poke_rewarded', 'poke_unrewarded', 'head_direction', 'goal_progress', 'goal_progress_distance', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'time_since_A', 'time_to_A', 'progress_since_A', 'distance_from_reward', 'distance_to_reward'], parameterization=reference_coded)
  Design rank: 160/160


Processing recording days:  20%|██        | 1/5 [00:44<02:57, 44.30s/it]

  CV: 6 folds, 16 groups, 0 perms — 35.4s  median r2_cv=-0.0029

ah10_20250613_20250615
  Design matrix: 29193 rows × 160 cols (regressors: ['place', 'task_state', 'poke_rewarded', 'poke_unrewarded', 'head_direction', 'goal_progress', 'goal_progress_distance', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'time_since_A', 'time_to_A', 'progress_since_A', 'distance_from_reward', 'distance_to_reward'], parameterization=reference_coded)
  Design rank: 160/160


Processing recording days:  40%|████      | 2/5 [01:23<02:04, 41.48s/it]

...

  CV: 6 folds, 16 groups, 0 perms — 7.6s  median r2_cv=-0.0081

ly06_20250613_20250615
  Design matrix: 13950 rows × 160 cols (regressors: ['place', 'task_state', 'poke_rewarded', 'poke_unrewarded', 'head_direction', 'goal_progress', 'goal_progress_distance', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'time_since_A', 'time_to_A', 'progress_since_A', 'distance_from_reward', 'distance_to_reward'], parameterization=reference_coded)
  Design rank: 160/160


Processing recording days:  80%|████████  | 4/5 [00:46<00:11, 11.27s/it]

  CV: 6 folds, 16 groups, 0 perms — 7.7s  median r2_cv=-0.0113

ly07_20250613_20250615
  Design matrix: 13621 rows × 160 cols (regressors: ['place', 'task_state', 'poke_rewarded', 'poke_unrewarded', 'head_direction', 'goal_progress', 'goal_progress_distance', 'speed', 'acceleration', 'time_from_reward', 'time_to_reward', 'time_since_A', 'time_to_A', 'progress_since_A', 'distance_from_reward', 'distance_to_reward'], parameterization=reference_coded)
  Design rank: 160/160


Processing recording days: 100%|██████████| 5/5 [00:57<00:00, 11.50s/it]

  CV: 6 folds, 16 groups, 0 perms — 8.3s  median r2_cv=0.0720
  500 ms / uniform   median r2_cv +0.02635


In [5]:
# Compare on delta_r2 (common denominator), not CPD
rows = []
for (width, scheme), res in results.items():
    for g in REG:
        v = np.nanmean([np.nanmedian(r['delta_r2_cv'][g]) for r in res.values()])
        rows.append({'width': width, 'scheme': scheme, 'regressor': g, 'delta_r2': v})
cmp_tbl = (pd.DataFrame(rows)
           .pivot_table(index='regressor', columns=['width', 'scheme'], values='delta_r2')
           .reindex(REG))
display(cmp_tbl.round(5))

print('\nheld-out r2_cv by configuration:')
for k, res in results.items():
    print(f'  {k}: {np.nanmean([np.nanmedian(v["r2_cv"]) for v in res.values()]):+.5f}')

width                       250               500         
scheme                   decile  uniform   decile  uniform
regressor                                                 
place                   0.00807  0.00843  0.01249  0.01258
task_state             -0.00129 -0.00143 -0.00210 -0.00227
poke_rewarded           0.00042  0.00141  0.00038  0.00160
poke_unrewarded         0.00105  0.00100  0.00151  0.00125
head_direction          0.00004  0.00071 -0.00072  0.00048
goal_progress          -0.00048 -0.00023 -0.00076 -0.00044
goal_progress_distance -0.00085 -0.00080 -0.00131 -0.00108
speed                   0.00128  0.00096  0.00234  0.00167
acceleration           -0.00009  0.00034 -0.00020  0.00021
time_from_reward       -0.00006 -0.00306 -0.00029 -0.00427
time_to_reward         -0.00095 -0.00308 -0.00167 -0.00478
time_since_A           -0.00149 -0.00320 -0.00233 -0.00487
time_to_A              -0.00131 -0.00280 -0.00220 -0.00434
progress_since_A       -0.00120 -0.00155 -0.00196 -0.00243
distance_from_reward   -0.00074 -0.00311 -0.00135 -0.00507
distance_to_reward     -0.00125 -0.00232 -0.00206 -0.00338


held-out r2_cv by configuration:
  (250, 'decile'): +0.02998
  (250, 'uniform'): +0.01811
  (500, 'decile'): +0.04133
  (500, 'uniform'): +0.02635


**Reading it.** Quantile placement fits the skewed reward-relative variables better
(`distance_from_reward` was +0.00045 quantile vs −0.00062 uniform in the standalone
comparison), but it moves shared variance: `goal_progress` came back −0.0003 under quantile
and **+0.0009** under uniform. That is not goal_progress encoding better — its own binning is
equal-width either way — it is uniform binning handicapping its competitors. **Any result that
flips between these columns is not robust and must be reported as such.**

## 2. Production fit

Driven from here. `RUN_MODE` picks how the 25 recdays get fitted:

- `'slurm'`   — submit one job per recday, poll until they finish, then merge. ~11 min/recday
  in parallel, so ~15 min wall clock.
- `'local'`   — fit them in this process, one after another. ~4.5 h; use for a couple of
  recdays, not the cohort.
- `'load'`    — skip fitting and load an already-merged result.

**Why shards rather than one pickle:** `run_or_load_glm` writes a single pickle per section
holding every recday, so 25 parallel jobs would race on that path and the loser's recdays
would vanish *silently* — the file still loads, just with fewer keys. Each job therefore
writes `{section}__{recday}__{artifact}.pkl` and the merge combines them, refusing to
proceed if a recday appears in two shards.

In [ ]:
RUN_MODE = 'load'                                 # 'load' reads the cached fit; set 'slurm' to REFIT (submits 25 jobs)
WIDTH, SCHEME, REGSET = 250, 'decile', 'full'      # <- set from section 1
SECTION = 'all_regressors'
PERMUTATIONS, CV_PERMS = 100, 100

sect = w.section_name(SECTION, width_ms=WIDTH, scheme=SCHEME, regset=REGSET)
cfg_flags = ['--section', SECTION, '--width-ms', str(WIDTH),
             '--scheme', SCHEME, '--regset', REGSET,
             '--permutations', str(PERMUTATIONS), '--cv-perms', str(CV_PERMS)]
print('section:', sect)
print('flags  :', ' '.join(cfg_flags))

In [ ]:
import subprocess, time, shlex

def _submit_and_wait(flags, poll_s=60):
    """Submit one job per recday and block until the queue drains."""
    cmd = ['bash', 'sbatch_files/submit_glm_lec.sh'] + flags
    print('$', ' '.join(shlex.quote(c) for c in cmd))
    print(subprocess.run(cmd, capture_output=True, text=True, cwd='..').stdout)
    while True:
        q = subprocess.run(['squeue', '--me', '--noheader', '--name'] +
                           [','.join(f'glm_{rd}' for rd in sorted(data_dic))],
                           capture_output=True, text=True).stdout.strip()
        n = len([l for l in q.split('\n') if l.strip()])
        print(f'  {n} job(s) still queued/running', flush=True)
        if n == 0:
            break
        time.sleep(poll_s)

if RUN_MODE == 'slurm':
    _submit_and_wait(cfg_flags)
    subprocess.run([sys.executable, 'code/run_glm_batch.py', '--merge'] + cfg_flags,
                   cwd='..', check=True)

elif RUN_MODE == 'local':
    import argparse
    ap = argparse.ArgumentParser(); rb.add_config_args(ap)
    args = ap.parse_args(cfg_flags)
    for rd in sorted(data_dic):
        rb.fit_one(rd, args)
    rb.merge(args)

elif RUN_MODE != 'load':
    raise ValueError(f'RUN_MODE must be slurm/local/load, got {RUN_MODE!r}')

In [ ]:
# load_glm_results returns a DICT keyed by artifact name (not a tuple).
res = glm.load_glm_results(rb.DEFAULT_OUT, sect, apply_exclusions=True, verbose=True)
print('artifacts:', sorted(res))

glm_results  = res['glm_results']
cpd_results  = res['cpd_results']
cv_results   = res['cv_results']
perm_results = res.get('permutation_results')
print(f'{len(cv_results)} recdays with cross-validated results')

In [ ]:
# The join contract: tuned_dict rows are POSITIONS in the sorted key list, so a gap shifts
# every later neuron relative to Neuron_raw and breaks the join to unit_regions.
unit_regions = asp.load_unit_regions()
asp.assert_glm_keys_contiguous(glm_results, unit_regions, strict=True)
print(f'{len(glm_results)} recdays, keys contiguous — safe to join')

## 3. Regional split

Every region on its own first, contrasts second. Recdays are not independent replicates —
mice are.

In [ ]:
# per-neuron delta_r2 -> {recday: array}, one regressor at a time
def delta_r2_by_regressor(cv_results, regressor):
    return {rd: np.asarray(r['delta_r2_cv'][regressor], dtype=float)
            for rd, r in cv_results.items() if regressor in r['delta_r2_cv']}

REGRESSOR = 'place'
joined = asp.join_regions(delta_r2_by_regressor(cv_results, REGRESSOR),
                          unit_regions, value_name='delta_r2', strict=True)
print(f'{len(joined)} units across {joined.recday.nunique()} recdays')

counts, feas = asp.feasibility_table(joined, min_units=20)
display(feas)

In [ ]:
# Per-region panel FIRST -- a region with n=1 mouse still gets a row, labelled descriptive
report = asp.per_region_report(joined, 'delta_r2', statistic='median')
display(report[['group','n_mice','n_recdays','n_units',
                'effect_pooled_over_mice','ci_lo','ci_hi','evidence']].round(5))
for _, r in report.iterrows():
    tag = f"   [{r['single_mouse_caveat']}]" if r['single_mouse_caveat'] else ''
    print(f"  {r['group']:10s} per-mouse: {r['per_mouse']}{tag}")

In [ ]:
# Contrasts second, with mice as the resampling unit
a, b = asp.PRIMARY_CONTRAST
print('bootstrap (mice resampled):')
print(' ', asp.cluster_bootstrap(joined, 'delta_r2', a, b, statistic='median'))
print('\nwithin-recday permutation null:')
print(' ', asp.within_recday_permutation(joined, 'delta_r2', a, b,
                                         statistic='median', n_perm=10000))

In [ ]:
# Robustness: rate matching and the 50 um boundary margin. A result that moves under
# either is unstable, not a finding.
scales = {rd: np.array([cpd_results[rd][n].get('__r2_full__', np.nan)
                        for n in sorted(cpd_results[rd])]) for rd in cpd_results}
jb = asp.boundary_margin_filter(joined, margin_um=50.0)
print(f'{jb.near_boundary.sum()}/{len(jb)} units within 50 um of a region transition')
print('\nexcluding boundary units:')
print(' ', asp.cluster_bootstrap(jb[~jb.near_boundary], 'delta_r2', a, b, statistic='median'))

## 4. All regressors by region

The table the project exists to produce. Δr² is the effect size (common denominator, so
regressors are comparable); the CV permutation p is the significance.

In [ ]:
rows = []
for g in REG:
    try:
        j = asp.join_regions(delta_r2_by_regressor(cv_results, g), unit_regions,
                             value_name='d', strict=False)
    except Exception as exc:
        print(f'  {g}: skipped ({exc})'); continue
    rep = asp.per_region_report(j, 'd', statistic='median')
    for _, r in rep.iterrows():
        rows.append({'regressor': g, 'group': r['group'],
                     'delta_r2': r['effect_pooled_over_mice'],
                     'n_mice': r['n_mice']})
tbl = (pd.DataFrame(rows)
       .pivot_table(index='regressor', columns='group', values='delta_r2')
       .reindex(REG)[[c for c in asp.ANALYSIS_GROUPS if c in set(pd.DataFrame(rows).group)]])
display(tbl.round(5))

## 5. LEC on its own terms, and by region

Sections 3–4 split by anatomy. This section asks what the LEC fit *is* — first pooled, then
regionally. **Run 5.1 before trusting any regional split:** if the model does not predict
held-out data, splitting it by region only partitions noise.

The first five panels are the same functions used on PFC
(`mFC_data/code/pfc_glm_plots.py`), byte-identical, so the two datasets are plotted the same
way. The rest are LEC-only, because PFC has no anatomy.

In [ ]:
import glm_plots as P
from importlib import reload; reload(P)

FIGDIR = '../data/figures/glm_production'
unit_regions = asp.load_unit_regions()
print(P.regressor_names(cv_results))

### 5.1 Does the model explain LEC at all?

### How a $\Delta R^2$ bar is computed

Five steps, from the fit to the bar height. They are not interchangeable with the obvious
alternatives, so the exact chain matters.

**1. Per neuron, per fold** — `glm_cv.cv_scores`, leave-one-session-out. Sessions are
different *tasks*, so a fold tests generalisation across tasks. Each neuron's firing is
mean-centred **within session** first (`cv_center_within_sessions=True`), which grants a free
per-session offset so the CV tests tuning shape rather than absolute-rate stability. For each
held-out session, the full model and each reduced model (one regressor group dropped) are fit
on the *other* sessions and scored on the held-out one.

**2. Per neuron — accumulate, then divide once.** RSS and TSS are summed across folds and the
ratio is formed at the end:

$$\Delta R^2_g \;=\; \frac{\sum_{\text{folds}} \mathrm{RSS}_{\text{reduced},g} \;-\; \sum_{\text{folds}} \mathrm{RSS}_{\text{full}}}{\sum_{\text{folds}} \mathrm{TSS}}$$

This is **not** the mean of per-fold $\Delta R^2$. Pooling before the ratio is more stable,
and it matters here because sessions differ in length so folds differ in size.

Two consequences worth holding onto:

- $\Delta R^2$ can be **negative** — dropping a regressor can *improve* held-out prediction
  when it was only fitting noise. That is signal, and is not clipped.
- The denominator is **TSS, shared by every regressor**, so bars are comparable to each other
  and roughly additive toward the model's $R^2_{cv}$. CPD instead divides by each group's own
  reduced-model RSS, which inflates weak regressors when a dominant one stays in the model —
  by up to 12$\times$ in simulation, which is exactly this dataset's situation (place
  dominates).

**3. Per recday** — median across that recday's neurons.

**4. Per mouse** — mean across that mouse's recdays.

**5. Bar height** — mean across mice. The dots are the step-4 values, one per mouse.

Steps 3–4 are deliberately two steps (`per_mouse_stat`). Recdays of one mouse are the same
probe in the same brain, re-sorted, so they are repeated measures rather than replicates — and
they carry very different neuron counts (1 to 117). Pooling all of a mouse's neurons into one
median would let its biggest recday dominate. On this fit the two schemes differ by up to
33% (`acceleration`), 22% (`speed`), and `goal_progress` changes sign.

This is the same chain as `anatomy_split.per_mouse_effect`, which the regional panels use —
verified identical to 0.00e+00 — so the pooled and regional figures are computed the same way.


In [ ]:
P.plot_model_fit(cv_results, out_path=f'{FIGDIR}/lec_model_fit.pdf');
P.plot_regressor_ranking(cv_results, out_path=f'{FIGDIR}/lec_regressor_ranking.pdf')
display(P.summary_table(cv_results).round(5))

### 5.2 Per-neuron structure

Mixed or specialised? The heatmap is `RdBu_r` centered at zero because Δr² is signed —
blue means dropping that regressor *improved* held-out prediction for that neuron.

### The permutation null: why Freedman–Lane

**What we want to test.** For each neuron and each regressor *g*: *does g explain held-out
variance beyond what the other regressors already explain?* That "beyond the others" clause is
the whole content of a CPD or $\Delta R^2$ — they are **unique**-variance measures.

**Why shuffling the firing gets this wrong.** The obvious null is to circularly shift the
neuron's spike train and refit. But that destroys **every** regressor's relationship to firing,
not just *g*'s. So it tests the *global* hypothesis "nothing explains this neuron", which is a
different and much weaker question — and with `place` dominating, it is trivially rejectable
for reasons that have nothing to do with *g*.

The damage is concrete. Under the shuffle, no regressor explains anything, so the model's
**residual variance $\sigma^2$ is much larger** than in the real fit where place and speed have
absorbed variance. The full model always pays a parameter penalty of roughly $k\sigma^2$ for
its *k* extra columns — so under the shuffle that penalty is inflated, the null sits *further*
below zero than the observed value does, and the observed beats it almost every time.

Measured on synthetic data where *g* is **exactly null** while the other regressors carry real
signal (so the correct answer is 5%):

| null | frac p < 0.05, CPD | frac p < 0.05, $\Delta R^2$ |
|---|---|---|
| shuffle the firing | 0.133 | **1.000** |
| **Freedman–Lane** | 0.067 | **0.050** |
| permute *g*'s columns | 0.067 | 0.067 |

The shuffle null gives a **100% false-positive rate** on $\Delta R^2$. All three retain full
power (1.000) when *g* does carry signal, so this is a specificity failure, not a sensitivity
one.

**How Freedman–Lane works.** For each regressor *g*, per neuron:

1. Fit the **reduced** model — everything except *g* — to the real firing, giving a prediction
   $\hat{y}_{\text{red}}$ and residuals $e = y - \hat{y}_{\text{red}}$.
2. Circularly shift those residuals **within each session**, giving $e^*$.
3. Build surrogate data $y^* = \hat{y}_{\text{red}} + e^*$.
4. Run the **whole cross-validation** on $y^*$ — refit full and reduced on the training
   sessions, score on the held-out one — and compute the statistic exactly as for the real data.
5. Repeat 100 times; $p = (1 + \#\{\text{null} \ge \text{observed}\}) / (1 + n_{\text{perm}})$.

**Why that is the right null.** $y^*$ keeps the other regressors' structure intact (it is built
*from* their fit), so the surrogate data has the same residual variance as the real data and the
full model pays the **same** parameter penalty it pays on the real data. Only *g*'s relationship
to firing is destroyed. That is exactly the hypothesis a unique-variance measure is asking
about — and it is why the null lands at 0.050 instead of 1.000.

Two implementation details that matter:

- **Circular shifting, not free permutation.** Neighbouring time bins are strongly
  autocorrelated (the animal occupies a maze node for many bins). Freely permuting residuals
  would destroy that autocorrelation and make the null far too easy to beat. Shifting preserves
  it. Shifting happens **within session** for the same reason the design does — wrapping one
  task's residuals onto another task's regressors would be a different null again.
- **One shift set, drawn once**, reused across every fold and every model, so the null does not
  additionally average over shift noise.

**Cost.** Freedman–Lane needs a different $y^*$ per regressor, so it cannot share one permuted
trace across models the way the shuffle can — measured at ~246 min per LEC recday at
`n_perm=100`, against ~72 min for the shuffle. Run as one SLURM job per recday, so the wall
clock is a few hours rather than days.

In [ ]:
P.plot_significant_fraction(cv_results, out_path=f'{FIGDIR}/lec_significant_fraction.pdf');
P.plot_neuron_heatmap(cv_results, sort_by='place', out_path=f'{FIGDIR}/lec_neuron_heatmap.pdf');
P.plot_mixed_selectivity(cv_results, out_path=f'{FIGDIR}/lec_mixed_selectivity.pdf');

### 5.3 Unit quality by region — before any tuning result

Region is derived from a unit's max-amplitude channel, and depth drives amplitude, isolation
and yield. So a regional difference in any rate-dependent statistic can be a recording-quality
gradient wearing an anatomical label. On this cohort that is **not** a small effect —
SUB/ProS fires ~3× faster than ENTl-deep — and the direction is not consistent across mice
(ly06 inverts it). This panel is why every later regional claim needs the rate-matched
version alongside it.

In [ ]:
P.plot_quality_by_region(unit_regions, out_path=f'{FIGDIR}/lec_quality_by_region.pdf');

### 5.4 The fit, split by region

Every region on its own before any contrast. Points are **mice**, not recdays — the recdays
of one mouse share a probe and a brain, so they are repeated measures. The small number under
each region is its n mice; a region with 1 is descriptive only and can never ground a
contrast claim.

In [ ]:
P.plot_regressor_ranking_by_region(cv_results, unit_regions,
                                   out_path=f'{FIGDIR}/lec_ranking_by_region.pdf');

In [ ]:
for g in ['place', 'goal_progress', 'task_state', 'head_direction']:
    P.plot_region_report(cv_results, g, unit_regions,
                         out_path=f'{FIGDIR}/lec_{g}_by_region.pdf')

**Reading 5.4.** `head_direction` is worth particular attention: it is the regressor the
W0.2 fix made interpretable for the first time (every previous HD result in this repo was
noise), and the one binned aggregation rescued. It is also absent from PFC, so it appears
here and nowhere in the cross-dataset comparison.

A regional difference here is not yet a finding. It has to survive the rate matching and the
50 µm boundary margin in section 3 before it counts, and `ENTl-sup` (90% ah08) and `ENTm`
(63% ly07) are single-mouse claims wearing region labels whatever the panel shows.

### Both effect sizes: $\Delta R^2$ and CPD

Every plotting function takes `value='delta_r2_cv'` (default) or `value='cpd_cv'`. They share
a numerator — held-out $\Delta$RSS — and differ only in denominator:

$$\Delta R^2_g = \frac{\Delta \mathrm{RSS}_g}{\mathrm{TSS}} \qquad\qquad
\mathrm{CPD}_g = \frac{\Delta \mathrm{RSS}_g}{\mathrm{RSS}_{\text{reduced},g}}$$

So $\mathrm{CPD} \ge \Delta R^2$ always, and the gap grows with how well the reduced model
still fits.

**On this data the choice barely matters, and I over-warned about it earlier.** The 12×
inflation I demonstrated in simulation needs a model that explains a lot; here $R^2_{cv}$ is
0.03–0.09, so $\mathrm{RSS}_{\text{reduced}} \approx \mathrm{TSS}$ and the two agree to within
about 8% (place: 0.01283 vs 0.01380 in LEC). Use $\Delta R^2$ for comparing regressors — the
common denominator is still the principled choice, and it becomes the *only* defensible one if
the fit ever improves — but no conclusion here turns on it.

**What does not change with `value`: the significance test.** `p_cv` is computed on CPD
regardless, so `frac_sig` is identical in both tables. See the significance section above for
why that is a mismatch worth fixing.

In [ ]:
for val in P.VALUE_OPTIONS:          # raw AND bias-corrected
    short = P._VALUE_SHORT[val]
    P.plot_regressor_ranking(cv_results, value=val,
                             out_path=f'../data/figures/glm_production/lec_ranking_{short}.pdf')
    P.plot_neuron_heatmap(cv_results, value=val,
                          out_path=f'../data/figures/glm_production/lec_heatmap_{short}.pdf')
    print(f'--- {short} ---')
    display(P.summary_table(cv_results, value=val).round(5))

The two side by side, per neuron per regressor. The dashed line is $y=x$;
distance above it is the inflation. Colour is regressor family.

In [ ]:
P.plot_cpd_vs_delta_r2(cv_results, out_path=f'../data/figures/glm_production/lec_cpd_vs_delta_r2.pdf');

### Bias-corrected effect sizes

Held-out $\Delta R^2$ and CPD both carry a **downward penalty proportional to a regressor's
column count**. The full model has *k* more parameters than the reduced one; if the regressor
explains nothing those *k* parameters fit training noise, and noise-fitted parameters *add*
held-out error. So a regressor explaining nothing scores about $-k\sigma^2/\text{denominator}$
— **not zero**.

Measured here: `corr(n_cols, null centre) = -0.838`, and all 16 nulls sit below zero.

**Zero is not the reference; the null centre is.** The permutation null measures that penalty
empirically per neuron (under permutation the regressor explains nothing by construction), so
subtracting it gives an unbiased effect:

| | under H₀ | with signal *s* |
|---|---|---|
| observed | $-k\sigma^2/\text{den}$ | $(s-k\sigma^2)/\text{den}$ |
| null mean | $-k\sigma^2/\text{den}$ | $-k\sigma^2/\text{den}$ |
| **corrected** | **≈ 0** | **≈ $s/\text{den}$** |

Correcting moves **14 of 16 regressors** in the ranking. `head_direction` (35 columns, the
largest penalty in the design) rises from 4th to 2nd; `poke_rewarded` (1 column, almost no
penalty) falls from 5th to 10th.

Two limits worth holding onto. It removes the *parameter penalty*, not the **capacity
advantage** — `corr(n_cols, corrected) = +0.517` remains, and a 35-column block genuinely can
capture more real structure than a 1-column indicator. And the correction inherits whichever
null it is centred on: these fits carry only the legacy shuffle-null key, so `resolve_value`
falls back to that. All three nulls measured the same centre on synthetics, but the
Freedman-Lane refit is what the plan specifies.

In [ ]:
import glm_analysis_v2 as _g, w1_refit as _w
_groups, _ = _g._resolve_regressor_groups(_w.SECTIONS['all_regressors']['regressors'],
                                          gp_n_bins=10, parameterization='reference_coded')
n_cols = {k: len(v) for k, v in _groups.items()}

P.plot_bias_correction(cv_results, n_cols=n_cols,
                       out_path=f'../data/figures/glm_production/lec_bias_correction.pdf');
display(P.summary_table(cv_results, value='delta_r2_corrected').round(5))